# Check and run an existing PyTorch model on the NPU

> **Community example.** Not maintained or supported by Q.ANT. See [AUTHORS.md](AUTHORS.md) and the [community README](../../README.md).

You already have a trained PyTorch model. Before you rewrite anything, you want three answers:

1. Which parts of it can run on the NPU, and which stay on the host?
2. How much data crosses the PCIe link in one forward pass?
3. How much accuracy do you lose in bfloat16?

`npu_compat` answers all three for an unmodified `nn.Module`. It traces the model with `torch.fx`, maps every node to a toolkit function where one exists, and leaves the rest in PyTorch so the model always runs end to end.

This notebook runs on the CPU backend. It checks compatibility and accuracy, not speed.

In [ ]:
import npu_compat
import small_cnn
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## The model you already have

`SmallCNN` is an ordinary MNIST classifier written with no knowledge of the NPU: two convolution blocks with batch norm, ReLU and max pooling, then two linear layers. Notice that `forward()` uses functional calls (`F.relu`, `torch.flatten`). A converter that only walks `model.children()` would miss them.

The first run trains it for one epoch on the CPU (about a minute) and saves `small_cnn.pt`. Later runs load the weights.

In [ ]:
model = small_cnn.load_or_train()
images, labels = small_cnn.test_batch(1000)

with torch.no_grad():
    fp32_logits = model(images)
fp32_accuracy = (fp32_logits.argmax(1) == labels).float().mean().item()
print(f"float32 accuracy on 1,000 test images: {fp32_accuracy:.1%}")

## Step 1: See where every layer runs

`convert()` traces the model with one example input, uploads the weights of the linear and convolution layers to the card once, and builds a plan. `report()` prints it.

In [ ]:
npu_model = npu_compat.convert(model, images[:1])
print(npu_model.report())

How to read it:

- **runs on** is taken from the toolkit v2.4.0 source. Linear and convolution layers do their matrix multiplications on the NPU. Batch norm splits: normalize and shift on the host, the scale step on the NPU. Bias, ReLU, pooling and softmax are toolkit functions that run on the host CPU.
- **Switches between NPU and host** counts every hand-over in one forward pass. Each one is a trip over PCIe, which the maintainers name as the bottleneck today.
- **Data to and from the card** adds up the features sent and the results read back. Weights are not counted because they were uploaded once. A convolution unfolds its input into patches on the host before uploading, so it moves more data than its input size suggests.
- **Weights resident on the card** is checked against a budget of about 17 GB per card. The driver enforces that limit on hardware, but the CPU backend doesn't, so the tool checks it for you. Lower it with `weight_budget_gb` to see what happens when a model doesn't fit.

## Step 2: Run it through the toolkit

`npu_model` is callable like the original. Supported layers go through the toolkit in bfloat16. Everything else runs in PyTorch.

In [ ]:
npu_logits = npu_model(images)
npu_accuracy = (npu_logits.argmax(1) == labels).float().mean().item()
agreement = (npu_logits.argmax(1) == fp32_logits.argmax(1)).float().mean().item()

print(f"float32 accuracy        {fp32_accuracy:.1%}")
print(f"toolkit accuracy        {npu_accuracy:.1%}")
print(f"same prediction as fp32 {agreement:.1%}")

## Step 3: Find where the error builds up

`compare()` runs the float32 model and the toolkit model side by side and measures the relative error of every layer's output.

- **so far** is the error of the layer's output in the full toolkit run, so it includes everything upstream.
- **added here** feeds the layer the exact float32 input and measures only its own rounding. A large value points at the layer to look at first.

In [ ]:
comparison = npu_model.compare(images[:32])
print(comparison)

Every layer adds a fraction of a percent, which is what bfloat16's 7-bit mantissa predicts. Batch norm adds the most, because the toolkit computes it in three steps and rounds after each one.

## Step 4: Fold batch norm into the convolutions

In inference, batch norm is a fixed scale and shift per channel, so it can be folded into the weights and bias of the convolution before it. `fold_batchnorm=True` does that with `torch.fx` before planning.

In [ ]:
folded = npu_compat.convert(model, images[:1], fold_batchnorm=True)
print(folded.report())

In [ ]:
before, after = npu_model.report(), folded.report()
before_err = comparison.output_error
after_err = folded.compare(images[:32]).output_error

rows = [
    ("NPU calls per pass", f"{before.npu_calls}", f"{after.npu_calls}"),
    ("Switches NPU and host", f"{before.switches}", f"{after.switches}"),
    ("KB to and from the card", f"{before.transfer_bytes / 1024:.1f}", f"{after.transfer_bytes / 1024:.1f}"),
    ("Output error", f"{before_err:.2%}", f"{after_err:.2%}"),
]
print(f"{'':26}{'as trained':>12}{'folded':>10}")
for name, a, b in rows:
    print(f"{name:26}{a:>12}{b:>10}")

Same predictions, fewer trips over PCIe, and less rounding. This is the kind of change the report is meant to surface: you don't need to know the toolkit's internals to find it.

## Step 5: Try a model with layers the toolkit doesn't cover

Real models contain layers with no toolkit function. They stay in PyTorch on the host, and the report lists them so you know what you would have to write yourself. Here is a small MLP block with LayerNorm, GELU and a residual add.

In [ ]:
class Block(nn.Module):
    def __init__(self, dim=32):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, 64)
        self.fc2 = nn.Linear(64, dim)

    def forward(self, x):
        return x + self.fc2(F.gelu(self.fc1(self.norm(x))))


class TinyMixer(nn.Module):
    def __init__(self):
        super().__init__()
        self.inp = nn.Linear(16, 32)
        self.blocks = nn.Sequential(Block(), Block())
        self.head = nn.Linear(32, 4)

    def forward(self, x):
        return F.softmax(self.head(self.blocks(self.inp(x))), dim=-1)


mixer = TinyMixer()
x = torch.randn(8, 16)
mixer_npu = npu_compat.convert(mixer, x[:1])
print(mixer_npu.report())
print()
print(mixer_npu.compare(x))

## Step 6: Use it on your own model

```python
npu_model = npu_compat.convert(your_model, example_input)
print(npu_model.report())
print(npu_model.compare(a_few_real_inputs))
```

To share the result, or to draw it as a layer map, export it as JSON:

In [ ]:
data = npu_compat.export_json(npu_model, images[:32], "small_cnn_report.json")
print(f"wrote {len(data['report']['layers'])} layers to small_cnn_report.json")

## Limits

- The model has to be traceable with `torch.fx`. Data-dependent control flow in `forward()` is not.
- Convolutions need `groups=1`, `dilation=1` and the same padding and stride in both directions. Others stay in PyTorch, and the report says why.
- Transposed convolution, pooling, softmax and batch norm reject batched input in toolkit 2.4.0, so the tool calls them one image at a time.
- Byte counts are estimates from reading the toolkit source, not measurements. On hardware, use `get_perf_counter` to measure.
- The CPU backend emulates the arithmetic, so timings here say nothing about the NPU.